# Ⅱ. 데이터 전처리

- 목적: 분석 목적에 맞게 데이터 구조를 표준화하고 지표 컬럼을 사전 생성한 뒤, SQLite에 적재하도록 설계

| 단계 | 내용 |
|------|------|
| 1 | 라이브러리 & 데이터 로드 |
| 2 | 데이터 타입 정규화 |
| 3 | 중복 응답 제거 |
| 4 | 회차 오기재 교정 |
| 5 | 리커트 척도 숫자 변환 |
| 6 | 익명화 처리 |
| 7 | 사전·사후 매칭 |
| 8 | 영역 평균 점수 생성 |
| 9 | 불필요한 컬럼 제거 및 순서 정렬 |
| 10 | SQLite 저장 |

## 1. 라이브러리 & 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import sqlite3

### 시트 이름 확인

In [5]:
PATH = "../data/stage1_clean.xlsx"

xls = pd.ExcelFile(PATH)
print(xls.sheet_names)

['사전_정제', '사후_정제']


### 각각 불러오기

In [6]:
df_pre = pd.read_excel(PATH, sheet_name="사전_정제")
df_post = pd.read_excel(PATH, sheet_name="사후_정제")

print("사전 shape:", df_pre.shape)
print("사후 shape:", df_post.shape)

사전 shape: (98, 15)
사후 shape: (98, 19)


## 2. 데이터타입 정규화

In [9]:
# 응답시각 -> datetime 변환
# 응답일은 이미 이전 단계에 생성됨.

for df in [df_pre, df_post]:
    df["응답시각"] = pd.to_datetime(df["응답시각"])

## 3. 중복 응답 제거

- 동일 선택회차와 참여자명 -> 최신 응답만 유지

In [10]:
def remove_duplicates(df):
    before = len(df)

    df = (
        df.sort_values("응답시각")
        .drop_duplicates(subset=["선택회차", "참여자명"], keep="last")
        .reset_index(drop=True)
    )

    after = len(df)
    print("중복 제거 건수: ", before-after)

    return df

print("=== 사전 설문 중복 제거 ====")
df_pre = remove_duplicates(df_pre)

print("=== 사후 설문 중복 제거 ====")
df_post = remove_duplicates(df_post)

=== 사전 설문 중복 제거 ====
중복 제거 건수:  7
=== 사후 설문 중복 제거 ====
중복 제거 건수:  7


In [12]:
# 중복 제거되었는지 확인용
display(df_pre.groupby(['선택회차', '참여자명']).size().sort_values(ascending=False))
display(df_post.groupby(['선택회차', '참여자명']).size().sort_values(ascending=False))

선택회차  참여자명
1회차   강은서     1
4회차   임민준     1
      오도윤     1
      안재원     1
      안유진     1
             ..
2회차   안지수     1
      안민재     1
      신수아     1
      송지수     1
5회차   황민재     1
Length: 91, dtype: int64

선택회차  참여자명
1회차   강은서     1
4회차   안재원     1
      안나은     1
      서혜원     1
      서재원     1
             ..
2회차   오태양     1
      안지수     1
      안민재     1
      신수아     1
5회차   황민재     1
Length: 91, dtype: int64

## 4. 회차 통합 - 정상회차 기반

In [13]:
print("사전 회차 오기재 건수:",
      (df_pre["선택회차"] != df_pre["정상회차"]).sum())

print("사후 회차 오기재 건수:",
      (df_post["선택회차"] != df_post["정상회차"]).sum())

사전 회차 오기재 건수: 4
사후 회차 오기재 건수: 4


In [14]:
# 최종 분석 기준 회차
df_pre["회차"] = df_pre["정상회차"]
df_post["회차"] = df_post["정상회차"]

## 5. 리커트 척도 숫자 변환

In [15]:
LIKERT_MAP = {
    "전혀 그렇지 않다": 1,
    "그렇지 않다": 2,
    "보통이다": 3,
    "그렇다": 4,
    "매우 그렇다": 5
}

likert_cols = [
    "A1_데이터정리수치도출",
    "A2_데이터시각화",
    "B1_업무활용계획",
    "B2_교육내용적용의향",
    "C1_팀내공유의향",
    "C2_데이터문화정착가능성"
]

def convert_likert(df):
    for col in likert_cols:
        df[col] = df[col].map(LIKERT_MAP)
    return df

df_pre = convert_likert(df_pre)
df_post = convert_likert(df_post)

## 6. 사전·사후 매칭

In [18]:
df_merged = pd.merge(df_pre, df_post,                  
    on=["회차", "참여자명"], 
    suffixes=('_pre', '_post'), 
    how="inner" #양쪽에 모두 존재하는 데이터만 남김
)

In [20]:
print("매칭 후 shape:", df_merged.shape)

매칭 후 shape: (86, 34)


In [23]:
df_merged.columns

Index(['응답시각_pre', '교육일_pre', '선택회차_pre', '참여자명', '직무_pre', '엑셀사용경력_pre',
       'A1_데이터정리수치도출_pre', 'A2_데이터시각화_pre', 'B1_업무활용계획_pre', 'B2_교육내용적용의향_pre',
       'C1_팀내공유의향_pre', 'C2_데이터문화정착가능성_pre', '응답일_pre', '정상회차_pre',
       '직무_정규_pre', '회차', '응답시각_post', '교육일_post', '선택회차_post', '직무_post',
       '엑셀사용경력_post', 'A1_데이터정리수치도출_post', 'A2_데이터시각화_post', 'B1_업무활용계획_post',
       'B2_교육내용적용의향_post', 'C1_팀내공유의향_post', 'C2_데이터문화정착가능성_post', 'D1_강사',
       'D2_내용', 'D3_운영환경', 'D4_전반만족도', '응답일_post', '정상회차_post', '직무_정규_post'],
      dtype='object')

In [21]:
df_merged.head(10)

,응답시각_pre,교육일_pre,선택회차_pre,참여자명,직무_pre,엑셀사용경력_pre,A1_데이터정리수치도출_pre,A2_데이터시각화_pre,B1_업무활용계획_pre,B2_교육내용적용의향_pre,...,B2_교육내용적용의향_post,C1_팀내공유의향_post,C2_데이터문화정착가능성_post,D1_강사,D2_내용,D3_운영환경,D4_전반만족도,응답일_post,정상회차_post,직무_정규_post
0,2025-03-05 01:13:19,2025-03-05,1회차,한채원,마케팅·광고·MD,1년 미만,3,3,2,3,...,4,4,5,그렇다,보통이다,매우 그렇다,그렇다,2025-03-05,1회차,마케팅
1,2025-03-05 01:22:50,2025-03-05,1회차,홍은서,디자인,거의 없음,3,1,2,1,...,2,2,4,그렇다,매우 그렇다,그렇다,보통이다,2025-03-05,1회차,디자인
2,2025-03-05 04:42:28,2025-03-05,1회차,강지수,마케팅/광고,3년 이상,4,4,5,5,...,4,4,4,매우 그렇다,그렇다,그렇다,보통이다,2025-03-05,1회차,마케팅
3,2025-03-05 07:43:04,2025-03-05,1회차,김소윤,마케팅·광고·MD,1~3년,3,4,3,2,...,4,4,4,그렇다,그렇다,보통이다,그렇다,2025-03-05,1회차,마케팅
4,2025-03-05 08:19:54,2025-03-05,1회차,홍혜원,기획·전략,거의 없음,1,1,2,3,...,4,4,3,매우 그렇다,그렇다,보통이다,보통이다,2025-03-05,1회차,기획/전략
5,2025-03-05 09:11:15,2025-03-05,1회차,조지호,마케팅,1~3년,3,2,3,3,...,4,4,4,그렇다,그렇다,그렇다,그렇다,2025-03-05,1회차,마케팅
6,2025-03-05 10:02:57,2025-03-05,1회차,장민서,디자인,3년 이상,2,4,3,3,...,4,5,4,보통이다,그렇다,그렇다,매우 그렇다,2025-03-05,1회차,디자인
7,2025-03-05 10:20:22,2025-03-05,1회차,정재원,디자인,3년 이상,4,3,3,3,...,4,4,4,그렇다,매우 그렇다,보통이다,매우 그렇다,2025-03-05,1회차,디자인
8,2025-03-05 10:53:49,2025-03-05,1회차,박주원,디자인,3년 이상,3,4,3,3,...,4,4,4,그렇다,그렇다,그렇다,보통이다,2025-03-05,1회차,디자인
9,2025-03-05 11:53:16,2025-03-05,1회차,홍소윤,전략기획,3년 이상,3,3,4,5,...,5,3,4,그렇다,그렇다,그렇다,그렇다,2025-03-05,1회차,기획/전략


## 7. 익명화 처리

In [24]:
# 기존 인덱스 초기화
df_merged = df_merged.reset_index(drop=True)

# 고유 id값 생성 - R001, R002
df_merged["id"] = [
    "R" + str(i+1).zfill(3) for i in range(len(df_merged))
]

# 참여자 실명 컬럼 삭제
df_merged = df_merged.drop(columns=["참여자명"])

In [26]:
df_merged.columns
"참여자명" in df_merged.columns

False

## 8. 영역 평균 점수 생성

- 아래 방식으로 구하기 
```
df_merged["사전_A"] = ( df_merged["A1_데이터정리수치도출_pre"] + df_merged["A2_데이터시각화_pre"] ) / 2 

df_merged["사후_A"] = ( df_merged["A1_데이터정리수치도출_post"] + df_merged["A2_데이터시각화_post"] ) / 2
```
- 문항 확장에도 코드 수정 없이 대응 가능하도록 설계

In [27]:
areas = ["A", "B", "C"]
time_map = {"pre": "사전", "post": "사후"}

for t_eng, t_kor in time_map.items():
    for area in areas:
        
        # 해당 영역 + 시점 컬럼 찾기
        cols = [
            c for c in df_merged.columns
            if c.startswith(area) and c.endswith(f"_{t_eng}")
        ]
        
        # 평균 생성
        df_merged[f"{t_kor}_{area}"] = df_merged[cols].mean(axis=1)

In [28]:
for t_kor in ["사전", "사후"]:
    
    area_cols = [f"{t_kor}_{a}" for a in areas]
    
    df_merged[f"{t_kor}_평균"] = df_merged[area_cols].mean(axis=1)

In [31]:
[col for col in df_merged.columns if ('사전' in col) or ('사후' in col)]

['사전_A', '사전_B', '사전_C', '사후_A', '사후_B', '사후_C', '사전_평균', '사후_평균']

## 9. 불필요한 컬럼 제거 및 순서 정렬

In [33]:
df_merged.columns

Index(['응답시각_pre', '교육일_pre', '선택회차_pre', '직무_pre', '엑셀사용경력_pre',
       'A1_데이터정리수치도출_pre', 'A2_데이터시각화_pre', 'B1_업무활용계획_pre', 'B2_교육내용적용의향_pre',
       'C1_팀내공유의향_pre', 'C2_데이터문화정착가능성_pre', '응답일_pre', '정상회차_pre',
       '직무_정규_pre', '회차', '응답시각_post', '교육일_post', '선택회차_post', '직무_post',
       '엑셀사용경력_post', 'A1_데이터정리수치도출_post', 'A2_데이터시각화_post', 'B1_업무활용계획_post',
       'B2_교육내용적용의향_post', 'C1_팀내공유의향_post', 'C2_데이터문화정착가능성_post', 'D1_강사',
       'D2_내용', 'D3_운영환경', 'D4_전반만족도', '응답일_post', '정상회차_post', '직무_정규_post',
       'id', '사전_A', '사전_B', '사전_C', '사후_A', '사후_B', '사후_C', '사전_평균', '사후_평균'],
      dtype='object')

### 필요한 컬럼만 추출해서 복사본 만들기

In [34]:
cols = [
    "id",
    "회차",
    "직무_정규_pre", 
    "엑셀사용경력_pre",
    "사전_A", "사전_B", "사전_C",
    "사후_A", "사후_B", "사후_C",
    "사전_평균", "사후_평균"
]

df_clean = df_merged[cols].copy()

### 컬럼 명칭 변경

In [35]:
df_clean = df_clean.rename(columns={
    "직무_정규_pre": "직무",
    "엑셀사용경력_pre": "엑셀사용경력"
})

In [37]:
df_clean.columns

Index(['id', '회차', '직무', '엑셀사용경력', '사전_A', '사전_B', '사전_C', '사후_A', '사후_B',
       '사후_C', '사전_평균', '사후_평균'],
      dtype='object')

- 필요한 컬럼들이 있음. 순서는 이미 정렬되어 있어서 추가로 하지 않음.

## 10. SQLite 저장

In [38]:
con = sqlite3.connect('../data/survey.db')

df_clean.to_sql("survey_clean", con, if_exists="replace", index=False)

con.close()

print("SQL로 저장 완료")

SQL로 저장 완료
